In [1]:
import os
import re
import math
import logging
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import fitz  # PyMuPDF


logging.basicConfig(format="%(levelname)s:%(message)s", level=logging.INFO)
logger = logging.getLogger(__name__)

_SOFT_HYPHEN = "\u00ad"
_DASH_CHARS = "-\u2010\u2011\u2012\u2013\u2014"

# Data structures

@dataclass
class Line:
    page: int
    block_no: int
    line_no: int
    x0: float
    y0: float
    x1: float
    y1: float
    text: str
    max_font: float
    flags: int


@dataclass
class Block:
    page: int
    block_no: int
    col_id: int
    kind: str   # HEADER | FOOTER | HEADING | PARA | SEPARATOR
    text: str
    x0: float
    y0: float
    x1: float
    y1: float

In [3]:
# Text helpers
def _norm_spaces(text: str) -> str:
    if not text:
        return ""
    text = text.replace(_SOFT_HYPHEN, "")
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def _is_all_capsish(text: str) -> bool:
    t = _norm_spaces(text)
    letters = re.sub(r"[^A-Za-z]", "", t)
    if not letters:
        return False
    upper = sum(1 for c in letters if c.isupper())
    return (upper / max(1, len(letters))) >= 0.88


def _starts_lowercase(text: str) -> bool:
    t = text.lstrip()
    return bool(t) and t[0].islower()


def _ends_sentence(text: str) -> bool:
    t = text.rstrip()
    return bool(t) and t[-1] in ".?!:"


def _ends_hyphen(text: str) -> bool:
    t = text.rstrip()
    return bool(t) and (t[-1] in _DASH_CHARS)


def _looks_like_word(text: str) -> bool:
    return bool(re.search(r"[A-Za-z]", text))


def _join_text(prev: str, nxt: str) -> str:
    prev = prev.rstrip()
    nxt = nxt.lstrip()
    if not prev:
        return nxt
    if not nxt:
        return prev
    if _ends_hyphen(prev):
        return prev[:-1] + nxt
    return prev + " " + nxt


In [4]:
# Text helpers

def _norm_spaces(text: str) -> str:
    if not text:
        return ""
    text = text.replace(_SOFT_HYPHEN, "")
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def _is_all_capsish(text: str) -> bool:
    t = _norm_spaces(text)
    letters = re.sub(r"[^A-Za-z]", "", t)
    if not letters:
        return False
    upper = sum(1 for c in letters if c.isupper())
    return (upper / max(1, len(letters))) >= 0.88

def _starts_lowercase(text: str) -> bool:
    t = text.lstrip()
    return bool(t) and t[0].islower()


def _ends_sentence(text: str) -> bool:
    t = text.rstrip()
    return bool(t) and t[-1] in ".?!:"


def _ends_hyphen(text: str) -> bool:
    t = text.rstrip()
    return bool(t) and (t[-1] in _DASH_CHARS)


def _looks_like_word(text: str) -> bool:
    return bool(re.search(r"[A-Za-z]", text))


def _join_text(prev: str, nxt: str) -> str:
    prev = prev.rstrip()
    nxt = nxt.lstrip()
    if not prev:
        return nxt
    if not nxt:
        return prev
    if _ends_hyphen(prev):
        return prev[:-1] + nxt
    return prev + " " + nxt

# Layout / heuristics

def _compute_body_font(lines: List[Line]) -> float:
    vals: Dict[float, int] = {}
    for ln in lines:
        t = _norm_spaces(ln.text)
        if not t:
            continue
        s = round(ln.max_font * 2) / 2
        vals[s] = vals.get(s, 0) + 1
    if not vals:
        return 10.0
    return float(max(vals.items(), key=lambda kv: kv[1])[0])


def _detect_column_anchors(page_lines: List[Line]) -> List[float]:
    """
    Detect dominant left-edge anchors. This is used only for assigning a block
    to a column band, not for reconstructing the reading order.

    Why this is safer for Congressional Record pages:
    - we keep PyMuPDF's block order as the primary reading order,
      because it already respects irregular first-page layouts fairly well;
    - anchors are only used to stop paragraph recombination from jumping
      across columns.
    """
    freq: Dict[int, int] = {}
    for ln in page_lines:
        t = _norm_spaces(ln.text)
        if not t:
            continue
        key = int(round(ln.x0 / 5.0) * 5)
        freq[key] = freq.get(key, 0) + 1

    common = sorted([x for x, c in freq.items() if c >= 4])
    if not common:
        common = sorted(freq.keys())

    merged: List[List[int]] = []
    for x in common:
        if not merged or abs(x - merged[-1][-1]) > 25:
            merged.append([x])
        else:
            merged[-1].append(x)

    anchors = [sum(g) / len(g) for g in merged]
    return anchors or [45.0]


def _nearest_anchor(x0: float, anchors: List[float]) -> int:
    if not anchors:
        return 0
    best_i = 0
    best_d = abs(x0 - anchors[0])
    for i, a in enumerate(anchors[1:], start=1):
        d = abs(x0 - a)
        if d < best_d:
            best_i, best_d = i, d
    return best_i


def _is_probable_heading(text: str, font: float, body_font: float) -> bool:
    t = _norm_spaces(text)
    if not t:
        return False
    if len(t) > 130:
        return False

    # Time markers are not headings.
    if re.match(r"^[bf]\s+\d{3,4}$", t, flags=re.I):
        return False
    if re.match(r"^\(?\d{1,2}:\d{2}\s*(a\.m\.|p\.m\.)", t, flags=re.I):
        return False

    # Large-font headings.
    if font >= body_font + 1.5:
        return True

    # Stacked all-caps headings, including broken title lines.
    if _is_all_capsish(t) and len(t) <= 95:
        return True

    # Section-style headings.
    if re.match(r"^(SEC\.|TITLE\s+[IVXLC]+\b|AMENDMENT|MOTION TO RECOMMIT)", t, flags=re.I):
        return True

    return False


def _is_separator_or_noise(text: str, x0: float, x1: float, y0: float, y1: float) -> bool:
    t = _norm_spaces(text)
    width = x1 - x0
    height = y1 - y0

    if not t:
        return True

    # Decorative / OCR artefacts and separator traces.
    if re.fullmatch(r"[_\-—–~.=•·*]{2,}", t):
        return True
    if re.fullmatch(r"[fb]", t, flags=re.I):
        return True
    if re.fullmatch(r"[fb]\s+\d{3,4}", t, flags=re.I):
        return True

    # Very small non-word fragments.
    if len(t) <= 2 and not _looks_like_word(t):
        return True

    # Extremely narrow vertical marginal text should be dropped.
    if width < 10 and height > 80:
        return True

    return False

# Extraction

def extract_page_lines(page: fitz.Page, page_no: int) -> List[Line]:
    td = page.get_text("dict", sort=True)
    out: List[Line] = []
    block_counter = -1

    for b in td.get("blocks", []):
        if b.get("type", 0) != 0:
            continue
        block_counter += 1
        for line_no, ln in enumerate(b.get("lines", [])):
            spans = ln.get("spans", [])
            if not spans:
                continue
            txt = "".join(sp.get("text", "") for sp in spans)
            if not txt or not txt.strip():
                continue
            max_font = max(float(sp.get("size", 0.0)) for sp in spans) if spans else 10.0
            flags = 0
            for sp in spans:
                flags |= int(sp.get("flags", 0))
            x0, y0, x1, y1 = ln.get("bbox", (0, 0, 0, 0))
            out.append(Line(
                page=page_no,
                block_no=block_counter,
                line_no=line_no,
                x0=float(x0), y0=float(y0), x1=float(x1), y1=float(y1),
                text=txt,
                max_font=max_font,
                flags=flags,
            ))
    return out


def extract_lines(doc: fitz.Document) -> List[Line]:
    all_lines: List[Line] = []
    for pno in range(doc.page_count):
        all_lines.extend(extract_page_lines(doc.load_page(pno), pno + 1))
    return all_lines

# Header / footer detection

def detect_headers_and_footers(
    lines: List[Line],
    top_frac: float = 0.08,
    bottom_frac: float = 0.05,
) -> Tuple[Dict[int, List[str]], Dict[int, set], Dict[int, List[str]], Dict[int, set]]:
    page_max_y: Dict[int, float] = {}
    for ln in lines:
        page_max_y[ln.page] = max(page_max_y.get(ln.page, 0.0), ln.y1)

    top_candidates: Dict[int, List[str]] = {}
    bot_candidates: Dict[int, List[str]] = {}

    for ln in lines:
        max_y = page_max_y.get(ln.page, 0.0)
        if max_y <= 0:
            continue
        t = _norm_spaces(ln.text)
        if not t:
            continue
        if ln.y0 <= max_y * top_frac:
            top_candidates.setdefault(ln.page, []).append(t)
        if ln.y1 >= max_y * (1.0 - bottom_frac):
            bot_candidates.setdefault(ln.page, []).append(t)

    def _repeated_keys(cands: Dict[int, List[str]]) -> Dict[str, int]:
        freq: Dict[str, int] = {}
        for _, items in cands.items():
            seen = set()
            for t in items:
                k = t.lower()
                if k in seen:
                    continue
                seen.add(k)
                freq[k] = freq.get(k, 0) + 1
        return freq

    top_freq = _repeated_keys(top_candidates)
    bot_freq = _repeated_keys(bot_candidates)

    top_pat = re.compile(r"(congressional record|^[A-Z]\d{3,6}$|^[A-Z][a-z]+ \d{1,2}, \d{4})", re.I)
    bot_pat = re.compile(r"(verdate|jkt\s+\d+|fmt\s+\d+|sfmt\s+\d+|\\CR\\FM\\|^[A-Z0-9]+PT\d+)", re.I)

    headers_text: Dict[int, List[str]] = {}
    header_keys: Dict[int, set] = {}
    footers_text: Dict[int, List[str]] = {}
    footer_keys: Dict[int, set] = {}

    for pg, items in top_candidates.items():
        uniq, keys, seen = [], set(), set()
        for t in items:
            k = t.lower()
            if k in seen:
                continue
            seen.add(k)
            if top_freq.get(k, 0) >= 2 or top_pat.search(t):
                uniq.append(t)
                keys.add(k)
        if uniq:
            headers_text[pg] = uniq
            header_keys[pg] = keys

    for pg, items in bot_candidates.items():
        uniq, keys, seen = [], set(), set()
        for t in items:
            k = t.lower()
            if k in seen:
                continue
            seen.add(k)
            if bot_freq.get(k, 0) >= 2 or bot_pat.search(t):
                uniq.append(t)
                keys.add(k)
        if uniq:
            footers_text[pg] = uniq
            footer_keys[pg] = keys

    return headers_text, header_keys, footers_text, footer_keys


# Build logical blocks

def build_blocks(
    lines: List[Line],
    header_keys: Dict[int, set],
    footer_keys: Dict[int, set],
) -> Tuple[List[Block], List[Tuple[int, str]], Dict[int, float], Dict[int, List[float]]]:
    lines_by_page: Dict[int, List[Line]] = {}
    for ln in lines:
        lines_by_page.setdefault(ln.page, []).append(ln)

    body_font_by_page: Dict[int, float] = {}
    anchors_by_page: Dict[int, List[float]] = {}
    for pg, page_lines in lines_by_page.items():
        body_font_by_page[pg] = _compute_body_font(page_lines)
        anchors_by_page[pg] = _detect_column_anchors(page_lines)

    all_blocks: List[Block] = []
    headings_list: List[Tuple[int, str]] = []

    for pg in sorted(lines_by_page.keys()):
        body_font = body_font_by_page[pg]
        anchors = anchors_by_page[pg]

        page_lines = sorted(lines_by_page[pg], key=lambda z: (z.block_no, z.line_no, z.y0, z.x0))
        blocks_by_no: Dict[int, List[Line]] = {}
        for ln in page_lines:
            blocks_by_no.setdefault(ln.block_no, []).append(ln)

        for block_no in sorted(blocks_by_no.keys()):
            blines = sorted(blocks_by_no[block_no], key=lambda z: (z.y0, z.x0))
            if not blines:
                continue

            col_id = _nearest_anchor(blines[0].x0, anchors)
            line_height = max(9.0, body_font * 1.2)
            gap_thresh = line_height * 1.2
            indent_tol = 18.0

            run_kind: Optional[str] = None
            run_text = ""
            run_lines: List[Line] = []
            prev_line: Optional[Line] = None

            def flush_run() -> None:
                nonlocal run_kind, run_text, run_lines, prev_line
                if run_kind and _norm_spaces(run_text):
                    x0 = min(z.x0 for z in run_lines)
                    y0 = min(z.y0 for z in run_lines)
                    x1 = max(z.x1 for z in run_lines)
                    y1 = max(z.y1 for z in run_lines)
                    b = Block(pg, block_no, col_id, run_kind, _norm_spaces(run_text), x0, y0, x1, y1)
                    all_blocks.append(b)
                    if run_kind == "HEADING":
                        headings_list.append((pg, b.text))
                run_kind = None
                run_text = ""
                run_lines = []
                prev_line = None

            for ln in blines:
                txt = _norm_spaces(ln.text)
                if not txt:
                    continue
                if txt.lower() in header_keys.get(pg, set()):
                    continue
                if txt.lower() in footer_keys.get(pg, set()):
                    continue
                if _is_separator_or_noise(txt, ln.x0, ln.x1, ln.y0, ln.y1):
                    flush_run()
                    all_blocks.append(Block(pg, block_no, col_id, "SEPARATOR", txt, ln.x0, ln.y0, ln.x1, ln.y1))
                    continue

                this_kind = "HEADING" if _is_probable_heading(txt, ln.max_font, body_font) else "PARA"

                if run_kind is None:
                    run_kind = this_kind
                    run_text = txt
                    run_lines = [ln]
                    prev_line = ln
                    continue

                if this_kind != run_kind:
                    flush_run()
                    run_kind = this_kind
                    run_text = txt
                    run_lines = [ln]
                    prev_line = ln
                    continue

                vgap = ln.y0 - (prev_line.y1 if prev_line else ln.y0)
                xshift = abs(ln.x0 - (prev_line.x0 if prev_line else ln.x0))

                if run_kind == "HEADING":
                    # Stack multi-line all-caps headings into one heading.
                    if vgap <= gap_thresh * 1.4:
                        run_text = _join_text(run_text, txt)
                        run_lines.append(ln)
                        prev_line = ln
                        continue
                    flush_run()
                    run_kind = this_kind
                    run_text = txt
                    run_lines = [ln]
                    prev_line = ln
                    continue

                # Paragraph merge.
                if _ends_hyphen(run_text) or (vgap <= gap_thresh and xshift <= indent_tol):
                    run_text = _join_text(run_text, txt)
                    run_lines.append(ln)
                    prev_line = ln
                else:
                    flush_run()
                    run_kind = this_kind
                    run_text = txt
                    run_lines = [ln]
                    prev_line = ln

            flush_run()

    return all_blocks, headings_list, body_font_by_page, anchors_by_page



# Recombine adjacent paragraph blocks

def recombine_paragraph_blocks(
    blocks_raw: List[Block],
    body_font_by_page: Dict[int, float],
) -> List[Block]:
    out: List[Block] = []
    i = 0

    while i < len(blocks_raw):
        b = blocks_raw[i]
        if b.kind != "PARA":
            out.append(b)
            i += 1
            continue

        body_font = body_font_by_page.get(b.page, 10.0)
        line_height = max(9.0, body_font * 1.2)
        join_gap = line_height * 1.5
        indent_tol = 20.0

        acc = Block(**b.__dict__)
        j = i + 1
        while j < len(blocks_raw):
            n = blocks_raw[j]
            if n.kind != "PARA":
                break
            if n.page != acc.page:
                break
            if n.col_id != acc.col_id:
                break

            gap = n.y0 - acc.y1
            indent_diff = abs(n.x0 - acc.x0)

            cont_cue = (
                _starts_lowercase(n.text)
                or not _ends_sentence(acc.text)
                or _ends_hyphen(acc.text)
            )
            similar_indent = indent_diff <= indent_tol
            small_gap = gap <= join_gap

            # Do not fuse paragraphs that look like a fresh speech line.
            speech_break = bool(re.match(r"^(Mr\.|Mrs\.|Ms\.|The SPEAKER|The Clerk)", n.text))
            if speech_break and _ends_sentence(acc.text):
                break

            if small_gap and (cont_cue or similar_indent):
                acc.text = _join_text(acc.text, n.text)
                acc.x0 = min(acc.x0, n.x0)
                acc.y0 = min(acc.y0, n.y0)
                acc.x1 = max(acc.x1, n.x1)
                acc.y1 = max(acc.y1, n.y1)
                j += 1
            else:
                break

        out.append(acc)
        i = j

    return out


# Matching

def windowed_matches(blocks: List[Block], phrase: str, window: int = 4) -> List[Tuple[int, str, List[Block]]]:
    needle = phrase.lower()
    out: List[Tuple[int, str, List[Block]]] = []
    for i in range(len(blocks)):
        parts: List[str] = []
        used: List[Block] = []
        start = blocks[i]
        for j in range(i, min(i + window, len(blocks))):
            b = blocks[j]
            if b.page != start.page:
                break
            if b.kind == "SEPARATOR":
                break
            parts.append(b.text)
            used.append(b)
            combined = _norm_spaces(" ".join(parts))
            if needle in combined.lower():
                out.append((b.page, combined, used.copy()))
                break
    return out


def matched_paragraph_blocks(blocks: List[Block], phrase: str) -> List[Block]:
    needle = phrase.lower()
    out: List[Block] = []
    seen = set()
    for b in blocks:
        if b.kind != "PARA":
            continue
        if needle in b.text.lower():
            key = (b.page, b.text)
            if key in seen:
                continue
            seen.add(key)
            out.append(b)
    return out


# Output

def _write_txt(path: str, content: str) -> None:
    os.makedirs(os.path.dirname(os.path.abspath(path)) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)


def _fmt_block(b: Block) -> str:
    return (
        f"--- PAGE {b.page} | BLOCK {b.block_no} | COL {b.col_id} | {b.kind} ---\n"
        f"{b.text}\n\n"
    )


# Main API

def process_pdf(
    pdf_path: str,
    out_dir: Optional[str] = None,
    match_phrase: str = "Endangered Species Act",
) -> Dict[str, str]:
    if not os.path.isfile(pdf_path):
        raise FileNotFoundError(f"File not found: {pdf_path}")

    if out_dir is None:
        out_dir = os.path.dirname(os.path.abspath(pdf_path)) or "."
    os.makedirs(out_dir, exist_ok=True)

    base = os.path.splitext(os.path.basename(pdf_path))[0]
    doc = fitz.open(pdf_path)

    try:
        lines = extract_lines(doc)
        headers_text, header_keys, footers_text, footer_keys = detect_headers_and_footers(lines)
        blocks_raw, headings, body_fonts, anchors = build_blocks(lines, header_keys, footer_keys)
        blocks_recombined = recombine_paragraph_blocks(blocks_raw, body_fonts)
        win_matches = windowed_matches(blocks_raw, match_phrase, window=4)
        para_matches = matched_paragraph_blocks(blocks_recombined, match_phrase)

        out_paths: Dict[str, str] = {}

        p_headers = os.path.join(out_dir, f"{base}_headers.txt")
        hdr_lines: List[str] = []
        for pg in sorted(headers_text.keys()):
            hdr_lines.append(f"=== PAGE {pg} HEADER ===")
            hdr_lines.extend(headers_text[pg])
            hdr_lines.append("")
        _write_txt(p_headers, "\n".join(hdr_lines).strip() + "\n")
        out_paths["headers"] = p_headers

        p_footers = os.path.join(out_dir, f"{base}_footers.txt")
        ftr_lines: List[str] = []
        for pg in sorted(footers_text.keys()):
            ftr_lines.append(f"=== PAGE {pg} FOOTER ===")
            ftr_lines.extend(footers_text[pg])
            ftr_lines.append("")
        _write_txt(p_footers, "\n".join(ftr_lines).strip() + "\n")
        out_paths["footers"] = p_footers

        p_layout = os.path.join(out_dir, f"{base}_layout_summary.txt")
        layout_lines: List[str] = []
        for pg in sorted(anchors.keys()):
            layout_lines.append(f"PAGE {pg}: body_font={body_fonts.get(pg, 10.0):.1f}, column_anchors={anchors[pg]}")
        _write_txt(p_layout, "\n".join(layout_lines) + "\n")
        out_paths["layout_summary"] = p_layout

        p_headings = os.path.join(out_dir, f"{base}_headings.txt")
        _write_txt(p_headings, "\n".join([f"[Page {p}] {h}" for p, h in headings]).strip() + "\n")
        out_paths["headings"] = p_headings

        p_blocks = os.path.join(out_dir, f"{base}_blocks.txt")
        _write_txt(p_blocks, "".join(_fmt_block(b) for b in blocks_raw if b.kind != "SEPARATOR"))
        out_paths["blocks"] = p_blocks

        p_recombined = os.path.join(out_dir, f"{base}_recombined_blocks.txt")
        _write_txt(p_recombined, "".join(_fmt_block(b) for b in blocks_recombined if b.kind == "PARA"))
        out_paths["recombined_blocks"] = p_recombined

        p_matches = os.path.join(out_dir, f"{base}_matches_{match_phrase.replace(' ', '_')}.txt")
        mlines = [
            f'MATCH PHRASE: "{match_phrase}"',
            f"TOTAL MATCHED WINDOWS: {len(win_matches)}",
            "",
        ]
        for pg, combined, used in win_matches:
            mlines.append(f"=== PAGE {pg} MATCH WINDOW ===")
            mlines.append(combined)
            mlines.append("---- SOURCE BLOCKS ----")
            for ub in used:
                mlines.append(f"[{ub.kind}] {ub.text}")
            mlines.append("")
        _write_txt(p_matches, "\n".join(mlines).strip() + "\n")
        out_paths["matches_windows"] = p_matches

        p_para = os.path.join(out_dir, f"{base}_matches_blocks_{match_phrase.replace(' ', '_')}.txt")
        _write_txt(p_para, "".join(_fmt_block(b) for b in para_matches))
        out_paths["matches_blocks"] = p_para

        logger.info("Done. Wrote outputs to: %s", out_dir)
        return out_paths
    finally:
        doc.close()

In [5]:
import os
from pathlib import Path

pdf_folder = Path("/Users/agnesnamyalo/Desktop/NEW_DOCS_copy/")  #Folder with the congressional pdf files

count = 0
for filename in os.listdir(pdf_folder):
    if filename.lower().endswith(".pdf"):
        #print(filename)
        count += 1

print("Total PDFs:", count)


Total PDFs: 249


In [6]:
import os
from pathlib import Path

if __name__ == "__main__":
    pdf_folder = Path("/Users/agnesnamyalo/Desktop/NEW_DOCS_copy/")

    for filename in os.listdir(pdf_folder):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(pdf_folder, filename)
            print(f"Processing: {pdf_path}")
            process_pdf(pdf_path, out_dir=None, match_phrase="Endangered Species Act")


Processing: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy/CREC-2014-03-13-pt1-PgH2385-2.pdf


INFO:Done. Wrote outputs to: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy


Processing: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy/CREC-2014-07-29-pt1-PgH7007-2.pdf


INFO:Done. Wrote outputs to: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy
INFO:Done. Wrote outputs to: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy
INFO:Done. Wrote outputs to: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy
INFO:Done. Wrote outputs to: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy


Processing: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy/CREC-1999-05-12-pt1-PgH3061-2.pdf
Processing: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy/CREC-2006-09-29-pt1-PgE1918.pdf
Processing: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy/CREC-1999-06-17-pt1-PgE1302-3.pdf
Processing: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy/CREC-2018-03-22-pt2-PgH2045-2.pdf


...

INFO:Done. Wrote outputs to: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy


Processing: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy/CREC-2017-11-01-pt1-PgH8326.pdf


INFO:Done. Wrote outputs to: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy


Processing: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy/CREC-2024-03-21-pt1-PgH1322.pdf


INFO:Done. Wrote outputs to: /Users/agnesnamyalo/Desktop/NEW_DOCS_copy


In [1]:
from pathlib import Path
import re
import os
from openai import OpenAI

# Folder containing my recombined block files
input_folder = Path("/Users/agnesnamyalo/Desktop/NEW_DOCS_copy_missing/")

#NEW_DOCS_copy_missing

# OpenAI client: set your key in terminal first
# export OPENAI_API_KEY="your-key-here"
#client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

client = OpenAI(api_key="sk-proj-")

esa_vocabulary = [
    "endangered species act",
    "esa",
    "ESA",
    "esa reform",
    "threatened species",
    "endangered species",
    "critical habitat",
    "listing",
    "delisting",
    "downlisting",
    "recovery plan",
    "recovery program",
    "fish and wildlife service",
    "national marine fisheries service",
    "listed species",
    "water",
    "species",
    "protection",
    "extinction"
]

def extract_blocks(text):
    blocks = []
    pattern = r'---\s*PAGE\s+(\d+)\s*\|\s*BLOCK\s+(\d+)\s*\|\s*COL\s+(\d+)\s*\|\s*PARA\s*---?\s*(.*?)(?=---\s*PAGE\s+\d+|$)'
    matches = re.finditer(pattern, text, re.DOTALL | re.IGNORECASE)

    for match in matches:
        page, block_num, col = match.groups()[:3]
        header = f"PAGE {page} | BLOCK {block_num} | COL {col} | PARA"
        body = match.group(4).strip()
        blocks.append((header, body))
    return blocks

def vocab_hit(full_text, esa_vocabulary):
    low = full_text.lower()
    return any(term in low for term in esa_vocabulary)

def is_esa_related_llm(full_text):
    vocab_as_text = "\n".join([f"- {term}" for term in esa_vocabulary])

    prompt = f"""
You are classifying congressional text for ESA relevance.

Classify as YES if this congressional text discusses ANY ESA-related topic OR contains one or more terms from the ESA vocabulary list in a way that is relevant to ESA.

ESA-related topics:
1. Endangered Species Act (ESA) explicitly mentioned.
2. ESA reform bills or legislation.
3. Fish and Wildlife Service ESA activities or decisions.
4. Species listing, delisting, downlisting, or recovery programs.
5. Endangered or threatened species management.
6. Habitat conservation or critical habitat under federal law.
7. Recovery programs involving listed species, such as the Upper Colorado River fish recovery program.
8. Direct Review of Protected Species Act or DROP Species Act.
9. ESA-related reform language even when ESA is not named directly.

ESA vocabulary:
{vocab_as_text}

YES examples:
- "Fish and Wildlife Service" plus species recovery programs.
- "Recovery Program" for listed species, such as Upper Colorado River.
- ESA reform bills by Mr. CALVERT.
- "ESA's 26 years" statistics.
- Text using ESA vocabulary in a clear ESA legal or policy context.

NO if the text is only about nature, wildlife, water, or conservation in a general way without a clear ESA connection.

Return ONLY YES or NO.

Text:
{full_text[:1500]}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=5,
            temperature=0
        )
        answer = response.choices[0].message.content.strip().upper()
        return answer == "YES"
    except Exception as e:
        print(f"LLM error: {e}")
        return False

def is_esa_related_hybrid(full_text):
    if vocab_hit(full_text, esa_vocabulary):
        return True
    return is_esa_related_llm(full_text)

header_pattern = re.compile(r"PAGE\s+(\d+)\s*\|\s*BLOCK\s+(\d+)\s*\|\s*COL\s+(\d+)", re.I)

def sort_key(header):
    m = header_pattern.search(header)
    if m:
        page, block, col = map(int, m.groups())
        return (page, col, block)
    return (999, 999, 999)

# Process all matching files
input_files = sorted(input_folder.glob("*_recombined_blocks.txt"))

print(f"Found {len(input_files)} files.")

for test_file in input_files:
    print(f"\nProcessing: {test_file.name}")

    text = test_file.read_text(encoding="utf-8", errors="ignore")
    blocks = extract_blocks(text)
    print(f"Total blocks: {len(blocks)}")

    esa_blocks = []
    for header, body in blocks:
        full_text = header + "\n" + body
        print(f"Checking: {header}")

        if is_esa_related_hybrid(full_text):
            esa_blocks.append((header, body))
            print(f"✅ {header}")
        else:
            print(f"❌ {header}")

    print(f"Found {len(esa_blocks)} ESA blocks in {test_file.name}")

    esa_blocks_sorted = sorted(esa_blocks, key=lambda x: sort_key(x[0]))
    output_lines = [h + "\n" + b + "\n\n" for h, b in esa_blocks_sorted]

    output_file = test_file.with_name(
        test_file.name.replace("_recombined_blocks.txt", "_ESA_ALL_new.txt")
    )
    output_file.write_text("".join(output_lines), encoding="utf-8")

    print(f"Saved -> {output_file.name}")

print("\nDone.")

Found 58 files.

Processing: CREC-2018-04-25-pt1-PgH3542_recombined_blocks.txt
Total blocks: 278
Checking: PAGE 1 | BLOCK 1 | COL 2 | PARA
❌ PAGE 1 | BLOCK 1 | COL 2 | PARA
Checking: PAGE 1 | BLOCK 3 | COL 2 | PARA
❌ PAGE 1 | BLOCK 3 | COL 2 | PARA
Checking: PAGE 1 | BLOCK 3 | COL 2 | PARA
❌ PAGE 1 | BLOCK 3 | COL 2 | PARA
Checking: PAGE 1 | BLOCK 3 | COL 2 | PARA
❌ PAGE 1 | BLOCK 3 | COL 2 | PARA
Checking: PAGE 1 | BLOCK 3 | COL 2 | PARA
❌ PAGE 1 | BLOCK 3 | COL 2 | PARA
Checking: PAGE 1 | BLOCK 4 | COL 0 | PARA
❌ PAGE 1 | BLOCK 4 | COL 0 | PARA
Checking: PAGE 1 | BLOCK 4 | COL 0 | PARA
❌ PAGE 1 | BLOCK 4 | COL 0 | PARA
Checking: PAGE 1 | BLOCK 4 | COL 0 | PARA
❌ PAGE 1 | BLOCK 4 | COL 0 | PARA
Checking: PAGE 1 | BLOCK 5 | COL 1 | PARA
❌ PAGE 1 | BLOCK 5 | COL 1 | PARA
Checking: PAGE 1 | BLOCK 6 | COL 0 | PARA
❌ PAGE 1 | BLOCK 6 | COL 0 | PARA
Checking: PAGE 2 | BLOCK 1 | COL 0 | PARA
❌ PAGE 2 | BLOCK 1 | COL 0 | PARA
Checking: PAGE 2 | BLOCK 1 | COL 0 | PARA
❌ PAGE 2 | BLOCK 1 | COL 

Cell above

input_folder.glob("*_recombined_blocks.txt") finds every matching file in the folder.

The script loops through each file automatically.

Each output keeps the same base filename but changes the suffix to _ESA_ALL_new.txt.

The sorting so it goes by page, col, block, which is usually more natural for reading congressional text than col, block, page.

The cell below version:

reads all _ESA_ALL_new.txt files from one folder,

removes the PAGE/BLOCK/COL headings,

counts the number of extracted ESA blocks,

creates a document_id by removing the _ESA_ALL_new suffix,

combines everything into one dataframe, one row per file,

and optionally saves a CSV.

In [2]:
from pathlib import Path
import pandas as pd
import re

# Folder containing your ESA text files
input_folder = Path("/Users/agnesnamyalo/Desktop/NEW_DOCS_copy/")


# Find all files ending with _ESA_ALL_new.txt
input_files = sorted(input_folder.glob("*_ESA_ALL_new.txt"))

heading_pattern = re.compile(
    r'^PAGE\s+\d+\s*\|\s*BLOCK\s+\d+\s*\|\s*COL\s+\d+\s*\|\s*PARA\s*\n?',
    re.MULTILINE
)

block_pattern = re.compile(
    r'^PAGE\s+\d+\s*\|\s*BLOCK\s+\d+\s*\|\s*COL\s+\d+\s*\|\s*PARA\s*$',
    re.MULTILINE
)

rows = []

for input_file in input_files:
    print(f"Processing: {input_file.name}")

    text = input_file.read_text(encoding="utf-8", errors="ignore")

    # Remove headings
    clean_text = heading_pattern.sub('', text).strip()

    # Count extracted ESA blocks
    total_blocks = len(block_pattern.findall(text))

    # Build clean document_id
    filename = input_file.stem
    document_id = re.sub(r'_ESA_ALL_new$', '', filename)

    rows.append({
        "document_id": document_id,
        "total_blocks": total_blocks,
        "extracted_text": clean_text
    })

# Create combined dataframe
df = pd.DataFrame(rows)

# save combined CSV
output_folder = Path("/Users/agnesnamyalo/Desktop/Research_work/")
output_file = output_folder / "ESA_ALL_new_combined_latest.csv"
df.to_csv(output_file, index=False, encoding="utf-8")

#print(f"\nDone. Processed {len(df)} files.")
#print(f"Saved combined CSV to: {output_file}")

Processing: CREC-1995-01-19-pt1-PgE134_ESA_ALL_new.txt
Processing: CREC-1995-02-23-pt1-PgH2072_ESA_ALL_new.txt
Processing: CREC-1995-02-23-pt1-PgH2087-4_ESA_ALL_new.txt
Processing: CREC-1995-02-27-pt1-PgE444-2_ESA_ALL_new.txt
Processing: CREC-1995-03-02-pt1-PgH2495-2_ESA_ALL_new.txt
Processing: CREC-1995-03-14-pt1-PgH3094-2_ESA_ALL_new.txt
Processing: CREC-1995-03-15-pt1-PgH3227-3_ESA_ALL_new.txt
Processing: CREC-1995-05-24-pt1-PgH5557_ESA_ALL_new.txt
Processing: CREC-1995-05-26-pt1-PgE1145-2_ESA_ALL_new.txt
Processing: CREC-1995-07-13-pt1-PgH6929-2_ESA_ALL_new.txt
Processing: CREC-1995-07-18-pt1-PgE1455-6_ESA_ALL_new.txt
Processing: CREC-1995-07-18-pt1-PgH7102-2_ESA_ALL_new.txt
Processing: CREC-1995-09-12-pt1-PgH8788_ESA_ALL_new.txt
Processing: CREC-1995-09-18-pt1-PgH9083_ESA_ALL_new.txt
Processing: CREC-1995-09-21-pt1-PgH9431_ESA_ALL_new.txt
Processing: CREC-1995-09-28-pt1-PgH9639-2_ESA_ALL_new.txt
Processing: CREC-1995-09-29-pt1-PgH9678-2_ESA_ALL_new.txt
Processing: CREC-1995-10-31-

In [3]:
from pathlib import Path
import pandas as pd
import re

df = pd.read_csv("/Users/agnesnamyalo/Desktop/Research_work/ESA_ALL_new_combined_latest.csv")

df.shape

(241, 3)